# 02 - Data Quality Inspection

- Inspecting the data quality of parquet file

In [50]:
# imports :
import os, sys
from pathlib import Path
from google.colab import drive
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

COLLAB = True
pd.set_option("display.max_columns", 100)

In [51]:
# Mount Drive :
if COLLAB :
  try:
      drive.mount("/content/drive")
  except Exception as e :                      # running outside Colab
      print(f"Error {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [52]:
BASE = Path("/content/drive/MyDrive/Projects/CSE-CIC-IDS2018")
RAW_DIR = BASE / "data"
PARQUET_PATH =  RAW_DIR / "processed" / "CSE_CIC_IDS2018_merged.parquet"
print("File exists:", PARQUET_PATH.exists())

File exists: True


# Step 1: Dataset overview

## 1.2 Size of the dataset

Read only the file’s metadata (nothing is loaded into RAM) to get the number of rows, columns and row groups.

In [53]:
pf = pq.ParquetFile(PARQUET_PATH)
meta = pf.metadata

print(f"Rows       : {meta.num_rows:,}")
print(f"Columns    : {meta.num_columns}")
print(f"Row groups : {meta.num_row_groups}")

Rows       : 16,233,002
Columns    : 80
Row groups : 88


## 1.3 Disk size and estimated memory

Show the size of the file on disk, and estimate the RAM the data would need as numbers. This justifies reading the data column by column later.

In [54]:
size_gb = PARQUET_PATH.stat().st_size / 1024**3
print(f"File size on disk: {size_gb:.2f} GB")

n_features = meta.num_columns - 2          # all columns except Timestamp and Label
est_gb = meta.num_rows * n_features * 4 / 1024**3
print(f"Estimated RAM if loaded as float32: {est_gb:.1f} GB")

File size on disk: 2.35 GB
Estimated RAM if loaded as float32: 4.7 GB


## 1.4 An extract

List the column names and the types stored in the file. All 80 should be string, as a result of the merge.

In [55]:
parquet_file = pq.ParquetFile(PARQUET_PATH)
df_preview = parquet_file.read_row_group(0).to_pandas().head(10000)

display(df_preview.head(3))

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443,6,02/03/2018 08:47:38,141385,9,7,553,3773,202,0,61.44444444,87.53443767,1460,0,539,655.4329358,30597.30523,113.1661775,9425.666667,19069.11685,73403,1,141385,17673.125,23965.32327,73403,22,51417,8569.5,13036.89082,31525,1,0,0,0,0,192,152,63.65597482,49.51020264,0,1460,254.4705882,474.7129551,225352.3897,0,0,1,1,0,0,0,1,0,270.375,61.44444444,539,0,0,0,0,0,0,9,553,7,3773,8192,119,4,20,0,0,0,0,0,0,0,0,Benign
1,49684,6,02/03/2018 08:47:38,281,2,1,38,0,38,0,19,26.87005769,0,0,0,0,135231.3167,10676.15658,140.5,174.655375,264,17,281,281,0,281,281,0,0,0,0,0,1,0,0,0,40,20,7117.437722,3558.718861,0,38,19,21.93931023,481.3333333,0,1,0,0,1,0,0,0,0,25.33333333,19,0,0,0,0,0,0,0,2,38,1,0,123,0,0,20,0,0,0,0,0,0,0,0,Benign
2,443,6,02/03/2018 08:47:40,279824,11,15,1086,10527,385,0,98.72727273,129.3924966,1460,0,701.8,636.3141856,41501.0864,92.91554692,11192.96,24379.44834,112589,1,279728,27972.8,36167.74032,112589,94,258924,18494.57143,36356.50372,133669,1,0,0,0,0,232,312,39.31042369,53.60512322,0,1460,430.1111111,566.234209,320621.1795,0,0,1,1,0,0,0,1,1,446.6538462,98.72727273,701.8,0,0,0,0,0,0,11,1086,15,10527,8192,1047,5,20,0,0,0,0,0,0,0,0,Benign


# Step 2: Target labels

## 2.1 Load only the Label column

Read the 16M labels without the 78 features. This is fast and light because Parquet stores each column separately.

In [56]:
labels = pq.read_table(PARQUET_PATH, columns=["Label"]).column("Label").to_pandas()

print(f"Labels loaded: {len(labels):,}")

Labels loaded: 16,233,002


## 2.2 Raw label values

**Classes found**
- The `Label` column contains **16 classes**: `Benign` and 14 attack types.
- One class `"Label"`, appears in **59 rows**. These are **repeated header rows** from the original CSV files, not real flows. They will be removed in the cleaning notebook.


**Class distribution**
- `Benign` dominates with **13,484,708 flows (≈ 83%)**;
- While the 14 attack types together represent **2,748,235 flows (≈ 17%)**.
- Grouped by attack family:

<div align="center">

| Family | Classes | Flows |
|---|---|---|
| DDoS | HOIC, LOIC-HTTP, LOIC-UDP | 1,263,933 |
| DoS | Hulk, SlowHTTPTest, GoldenEye, Slowloris | 654,300 |
| Brute force | FTP, SSH | 380,949 |
| Botnet | Bot | 286,191 |
| Infiltration | Infilteration | 161,934 |
| Web attacks | Brute Force -Web, Brute Force -XSS, SQL Injection | 928 |

</div>


In [57]:
raw_counts = labels.value_counts()

for label, n in raw_counts.items():
    print(f"{label!r:35} {n:>12,}")

'Benign'                              13,484,708
'DDOS attack-HOIC'                       686,012
'DDoS attacks-LOIC-HTTP'                 576,191
'DoS attacks-Hulk'                       461,912
'Bot'                                    286,191
'FTP-BruteForce'                         193,360
'SSH-Bruteforce'                         187,589
'Infilteration'                          161,934
'DoS attacks-SlowHTTPTest'               139,890
'DoS attacks-GoldenEye'                   41,508
'DoS attacks-Slowloris'                   10,990
'DDOS attack-LOIC-UDP'                     1,730
'Brute Force -Web'                           611
'Brute Force -XSS'                           230
'SQL Injection'                               87
'Label'                                       59


# Step 3: Data types

- In the merged parquet, **all 80 columns are stored as object**.
- This was to avoid data type conflicts between chunks during merging.


- In the original CSV files, the types were:
  - **42 integer** columns (packet counts, ports, flags, lengths)
  - **36 float** columns (means, standard deviations, rates such as `Flow Byts/s`)
  - **2 text** columns: `Timestamp` and `Label`

- The 78 (42 + 36) feature columns will be **converted to numbers** in the next step. Values that can't be
  converted (e.g. `"Infinity"`, `"NaN"`, empty strings, header rows) will be counted there.

In [58]:
print(df_preview.dtypes.value_counts())

object    80
Name: count, dtype: int64


# Step 4: Per-column report

Get the features columns, skiping `Label` (target) and `Timestamp`

In [59]:
feature_cols = [c for c in pf.schema_arrow.names if c not in ("Timestamp", "Label")]

labels = pq.read_table(PARQUET_PATH, columns=["Label"]).column("Label").to_pandas()
is_header = (labels == "Label").to_numpy()
del labels

print(f"Feature columns: {len(feature_cols)}")
print(f"Header rows to skip: {is_header.sum()}")

Feature columns: 78
Header rows to skip: 59


In [60]:
def column_report(col):
    values = pq.read_table(PARQUET_PATH, columns=[col]).column(col).to_pandas()
    values = values[~is_header]

    num = pd.to_numeric(values, errors="coerce")      # text that isn't a number -> NaN
    finite = num[np.isfinite(num)]                    # without NaN and Inf

    return {
        "missing": num.isna().sum(),
        "inf": np.isinf(num).sum(),
        "negative": (num < 0).sum(),
        "unique": finite.nunique(),
        **finite.describe().to_dict(),                # count, mean, std, min, 25%, 50%, 75%, max
    }

In [ ]:
rows = {}
for i, col in enumerate(feature_cols, 1):
    rows[col] = column_report(col)
    print(f"{i}/{len(feature_cols)} {col}")

report = pd.DataFrame(rows).T
report.to_csv( RAW_DIR / "quality_report_full.csv")
report.head()

In [62]:
report_path = RAW_DIR / "quality_report_full.csv"
report = pd.read_csv(report_path)

In [63]:
print("\n Columns with missing values:")
display(report.loc[report["missing"] > 0, ["missing"]].T)

print("\n Columns with infinite values:")
display(report.loc[report["inf"] > 0, ["inf"]].T)

print("\n Columns with negative values:")
display(report.loc[report["negative"] > 0, ["negative", "min"]].T)

print("\n Constant columns (a single value):")
display(pd.DataFrame(columns=report.index[report["unique"] <= 1]))


 Columns with missing values:


,15
missing,59721.0



 Columns with infinite values:


,15,16
inf,36039.0,95760.0



 Columns with negative values:


,2,16,17,19,20,21,22,24,25,66,67
negative,1.400000e+01,14.000000,1.400000e+01,3.000000e+00,1.500000e+01,1.400000e+01,1.400000e+01,3.000000e+00,1.500000e+01,4432608.0,8255549.0
min,-9.190110e+11,-0.008895,-8.282200e+11,-8.282200e+11,-9.474050e+11,-9.190110e+11,-8.282200e+11,-8.282200e+11,-9.474050e+11,-1.0,-1.0



 Constant columns (a single value):


,32,34,56,57,58,59,60,61


### Interpretation

**1. Missing and infinite values**

Only two columns are affected:
- `Flow Byts/s`: 59,721 missing values and 36,039 infinite values.
- `Flow Pkts/s`: 95,760 infinite values.

**2. Negative values**

- **Time-related features** (`Flow Duration`, `Flow IAT`, `Fwd IAT`, etc.): only 3–15 rows per feature contain negative values. These are invalid measurements, possibly caused by clock issues during traffic capture.

**3. Constant features**

Eight columns have the same value across all 16M rows. They provide no useful information for the model:
- `Bwd PSH Flags`, `Bwd URG Flags`
- `Fwd Byts/b Avg`, `Fwd Pkts/b Avg`, `Fwd Blk Rate Avg`
- `Bwd Byts/b Avg`, `Bwd Pkts/b Avg`, `Bwd Blk Rate Avg`


### Cleaning decisions

| Issue | Columns | Rows affected | Decision |
|---|---|---:|---|
| Missing / infinite values | `Flow Byts/s`, `Flow Pkts/s` | ≈ 95,760 (0.59%) | Remove affected rows |
| Negative time values | Time-related features | 3–15 per feature | Remove affected rows |
| `-1` placeholder | `Init Fwd/Bwd Win Byts` | 27% / 51% | Keep rows and treat `-1` as unavailable |
| Constant features | 8 columns | — | Remove columns |
| Repeated header rows | All columns | 59 | Remove affected rows |

# Step 5 : duplicated rows

- **4,229,016 duplicate rows (26%)** of the 16.2M flows: identical in all features and the label.
- Automated attacks repeat the same flow many times:

| Class | Rows | Duplicates | Unique rows left |
|---|---|---|---|
| FTP-BruteForce | 193,360 | 99.97% | ≈ 54 |
| DoS SlowHTTPTest | 139,890 | 99.96% | ≈ 55 |
| DDOS attack-HOIC | 686,012 | 71% | ≈ 198,900 |
| DoS attacks-Hulk | 461,912 | 69% | ≈ 145,200 |
| Bot / SSH-Bruteforce | 286,191 / 187,589 | ≈ 50% | ≈ 144,500 / 94,000 |
| Benign | 13,484,708 | 21% | ≈ 10.65M |

- Duplicates are a problem: the same row can end up in both train and test sets, which **inflates the scores**.

**Decision for cleaning:** drop the exact duplicates. FTP-BruteForce and SlowHTTPTest then become
**very small classes** (≈ 55 rows each) and will need to be merged or dropped.

In [64]:
hashes = []
for g in range(pf.num_row_groups):
    batch = pf.read_row_group(g).to_pandas().drop(columns=["Timestamp"])
    hashes.append(pd.util.hash_pandas_object(batch, index=False).to_numpy())

hashes = pd.Series(np.concatenate(hashes))[~is_header]     # skip the 59 header rows

In [65]:
dup = hashes.duplicated()
print(f"Duplicate rows: {dup.sum():,} ({100 * dup.mean():.2f}%)")

Duplicate rows: 4,229,016 (26.05%)


In [66]:
labels = pq.read_table(PARQUET_PATH, columns=["Label"]).column("Label").to_pandas()
labels[~is_header][dup].value_counts()

,count
Label,
Benign,2833418
DDOS attack-HOIC,487151
DoS attacks-Hulk,316713
FTP-BruteForce,193306
Bot,141656
DoS attacks-SlowHTTPTest,139835
SSH-Bruteforce,93541
Infilteration,21324
DoS attacks-Slowloris,1082


# Step 6: Conversion and sampling

The full dataset (16.2M rows) does not fit in Colab RAM as numbers, so we create a **working sample**
for the next notebooks:
- **all attack rows are kept** (100%)
- **15% of the Benign rows** are kept, chosen at random (seed 42); Benign stays the majority class
- the 78 features are converted from text to **`float32`**

Nothing else is changed here: NaN, Inf, duplicates and header rows are still in the sample.
They are removed in the cleaning notebook (03).

## 6.1 Convert and sample, one row group at a time
Each row group is read, sampled and converted, then added to a list. Only one row group of text is in memory at a time.

In [67]:
BENIGN_FRAC = 0.15
rng = np.random.default_rng(42)

parts = []
for g in range(pf.num_row_groups):
    batch = pf.read_row_group(g).to_pandas()

    keep = (batch["Label"] != "Benign") | (rng.random(len(batch)) < BENIGN_FRAC)
    batch = batch[keep].copy()

    batch[feature_cols] = batch[feature_cols].apply(pd.to_numeric, errors="coerce").astype("float32")
    parts.append(batch)
    print(f"{g + 1}/{pf.num_row_groups}")

df = pd.concat(parts, ignore_index=True)
del parts

1/88
2/88
3/88
4/88
5/88
6/88
7/88
8/88
9/88
10/88
11/88
12/88
13/88
14/88
15/88
16/88
17/88
18/88
19/88
20/88
21/88
22/88
23/88
24/88
25/88
26/88
27/88
28/88
29/88
30/88
31/88
32/88
33/88
34/88
35/88
36/88
37/88
38/88
39/88
40/88
41/88
42/88
43/88
44/88
45/88
46/88
47/88
48/88
49/88
50/88
51/88
52/88
53/88
54/88
55/88
56/88
57/88
58/88
59/88
60/88
61/88
62/88
63/88
64/88
65/88
66/88
67/88
68/88
69/88
70/88
71/88
72/88
73/88
74/88
75/88
76/88
77/88
78/88
79/88
80/88
81/88
82/88
83/88
84/88
85/88
86/88
87/88
88/88


## 6.2 Check the sample
Size, types and memory of the working sample.

In [68]:
print("Shape:", df.shape)
print(df.dtypes.value_counts())
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

Shape: (4769628, 80)
float32    78
object      2
Name: count, dtype: int64
Memory: 1.96 GB


## 6.3 Classes: full data vs sample
All attack classes should keep 100% of their rows, and Benign about 15%.

In [69]:
compare = pd.DataFrame({
    "full": labels.value_counts(),
    "sample": df["Label"].value_counts(),
}).fillna(0).astype(int)
compare["kept_%"] = (100 * compare["sample"] / compare["full"]).round(1)
compare

,full,sample,kept_%
Label,,,
Benign,13484708,2021334,15.0
DDOS attack-HOIC,686012,686012,100.0
DDoS attacks-LOIC-HTTP,576191,576191,100.0
DoS attacks-Hulk,461912,461912,100.0
Bot,286191,286191,100.0
FTP-BruteForce,193360,193360,100.0
SSH-Bruteforce,187589,187589,100.0
Infilteration,161934,161934,100.0
DoS attacks-SlowHTTPTest,139890,139890,100.0


## 6.4 Save the sample
This file is the input of the cleaning notebook (03).

In [70]:
SAMPLE_PATH = RAW_DIR / "processed" / "CSE_CIC_IDS2018_sample.parquet"
df.to_parquet(SAMPLE_PATH, index=False)

print("Saved:", SAMPLE_PATH)
print(f"File size: {SAMPLE_PATH.stat().st_size / 1024**2:.0f} MB")

Saved: /content/drive/MyDrive/Projects/CSE-CIC-IDS2018/data/processed/CSE_CIC_IDS2018_sample.parquet
File size: 350 MB


# Step 7: Summary of the data quality inspection

All counts below come from the **full dataset** (16,232,943 flows, header rows excluded).

| # | Problem | Where | Rows affected | Decision (notebook 03) |
|---|---|---|---|---|
| 1 | Data stored as text | All 78 features | All | Converted to `float32` (done in step 6) |
| 2 | Repeated header rows | All columns | 59 | Drop the rows |
| 3 | Duplicate rows | All columns | 4,229,016 (26%) | Drop the duplicates |
| 4 | Missing / infinite values | `Flow Byts/s`, `Flow Pkts/s` | ≈ 95,760 (0.59%) | Drop the rows |
| 5 | Negative time values | `Flow Duration`, IAT features | ≤ 15 | Drop the rows |
| 6 | `-1` placeholder | `Init Fwd/Bwd Win Byts` | 27% / 51% | Keep, `-1` = "not available" |
| 7 | Constant features | 8 columns | — | Drop the columns |
| 8 | `Timestamp` column | — | — | Drop (not a flow feature, would leak the capture day) |
| 9 | Class imbalance | `Label` | Benign ≈ 83% | Benign sampled (step 6), class weights, macro-F1 |
| 10 | Very small classes | Web attacks (928 rows), FTP-BruteForce & SlowHTTPTest (≈ 55 unique rows each) | — | Merge the web attacks; decide on FTP / SlowHTTPTest after removing duplicates |

**Output of this notebook:** `CSE_CIC_IDS2018_sample.parquet`, with all attack flows and 15% of the
Benign flows, features as `float32`, not cleaned yet.